In [9]:
%%writefile synth.py

"""
synth.py - Synthetic higher-order context-effect study (MTP, Ajaz Hussain).

Same file runs on Kaggle (import it in cells) and on the cluster (command line).
Data are regenerated deterministically from (dataset, seed), so every platform
sees identical data and no data files need to be copied around.

Datasets (first set of experiments, "pure" orders):
  D0_lcl    : LCL,  u_i = theta.x_i + lam * x_i' A ybar_S           (order 1, linear)
  D1_sim    : similarity, closeness of i to ONE other item           (order 1, nonlinear)
  D2_comp   : compromise, closeness of i to the MIDPOINT of TWO      (order 2)
  D3_centre : centre, closeness of i to the CENTRE of THREE          (order 3)
"""
import argparse, copy, itertools, json, math, os, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

# ------------------------------------------------------------------ settings
N_ITEMS, D = 10, 6            # items per slate, features per item
CTX = [0, 1]                  # the 2 features used for closeness ("price", "quality")
N_TRAIN, N_VAL, N_TEST = 285_000, 15_000, 50_000

DATASETS = {
    "D0_lcl":    dict(id=0, kind="lcl"),
    "D1_sim":    dict(id=1, kind="k", k=1, sigma=0.2, sign=-1.0),   # near-copy steals share
    "D2_comp":   dict(id=2, kind="k", k=2, sigma=0.2, sign=+1.0),   # middle option gains
    "D3_centre": dict(id=3, kind="k", k=3, sigma=0.2, sign=+1.0),
}
# Strength of the planted context effect, from Exp 0 (Cell 0.5), sigma = 0.2:
#   D0: twin gap 0.05 nats (control, effect is the slate mean)
#   D1-D3: part NOT explained by own + others' mean & sd = 0.03 nats in each
# Use the SAME values on every platform.
LAMBDA = {"D0_lcl": 1.3120, "D1_sim": 1.8310, "D2_comp": 0.5087, "D3_centre": 0.2507}


# ------------------------------------------------------------------ ground truth
def truth(ds, seed):
    """theta (and A for LCL) drawn per seed; ||theta|| = 1, ||A||_F = 1."""
    rng = np.random.default_rng([DATASETS[ds]["id"], seed, 1])
    theta = rng.normal(size=D); theta /= np.linalg.norm(theta)
    U, V = rng.normal(size=(D, 2)), rng.normal(size=(D, 2))
    A = U @ V.T; A /= np.linalg.norm(A)
    return theta, A


def context_raw(X, ds, A, chunk=10_000):
    """Planted context term c_i (before multiplying by lambda). X: (N, n, D)."""
    spec = DATASETS[ds]
    if spec["kind"] == "lcl":
        ybar = X.mean(1, keepdims=True)                       # mean of the whole slate
        return ((X @ A) * ybar).sum(-1)                       # x_i' A ybar
    k, s2 = spec["k"], spec["sigma"] ** 2
    combos = np.array(list(itertools.combinations(range(N_ITEMS), k)))   # (T, k)
    member = np.zeros((N_ITEMS, len(combos)), bool)
    for t, c in enumerate(combos):
        member[c, t] = True                                    # item i is inside combo t
    out = np.empty(X.shape[:2])
    for a in range(0, len(X), chunk):
        P = X[a:a + chunk][..., CTX]                           # (B, n, 2)
        cent = P[:, combos, :].mean(2)                         # (B, T, 2) midpoint / centre
        d2 = ((P[:, :, None, :] - cent[:, None, :, :]) ** 2).sum(-1)   # (B, n, T)
        K = np.exp(-d2 / s2) * (~member)[None]                 # only combos of OTHER items
        out[a:a + chunk] = K.sum(-1)
    return out


def make_data(ds, seed, lam=None, n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST):
    """Returns dict with X (N,n,D) float32, y (N,), P (true probs), U (utilities), splits."""
    lam = LAMBDA[ds] if lam is None else lam
    assert lam is not None, f"LAMBDA['{ds}'] not set - run Exp 0 first"
    theta, A = truth(ds, seed)
    rng = np.random.default_rng([DATASETS[ds]["id"], seed, 2])
    N = n_train + n_val + n_test
    X = rng.normal(size=(N, N_ITEMS, D))
    c = context_raw(X, ds, A)
    sign = DATASETS[ds].get("sign", 1.0)
    u = X @ theta + sign * lam * c
    P = np.exp(u - u.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
    y = (P.cumsum(1) < rng.random((N, 1))).sum(1).clip(max=N_ITEMS - 1)
    u0 = X @ theta                                             # context-free truth
    idx = {"train": slice(0, n_train), "val": slice(n_train, n_train + n_val),
           "test": slice(n_train + n_val, N)}
    return dict(X=X.astype(np.float32), y=y, P=P.astype(np.float32),
                U0=u0.astype(np.float32), c=c, idx=idx, theta=theta, A=A)


def oracle_metrics(data):
    """Best possible numbers (true model) and the context-free truth, on test."""
    t = data["idx"]["test"]; P, y = data["P"][t].astype(np.float64), data["y"][t]
    logP = np.log(P + 1e-30)
    U0 = data["U0"][t].astype(np.float64)
    logP0 = U0 - U0.max(1, keepdims=True); logP0 -= np.log(np.exp(logP0).sum(1, keepdims=True))
    return dict(test_nll=float(-logP[np.arange(len(y)), y].mean()),
                test_ce=float(-(P * logP).sum(1).mean()),            # = entropy
                plain_test_ce=float(-(P * logP0).sum(1).mean()))      # theta only, no context


# ------------------------------------------------------------------ Exp 0: calibration
def calibrate(ds, lams, seeds=(0, 1, 2), n_slates=20_000, target=0.05):
    """No training. For each lambda, estimate how much the planted context is worth
    to a no-context model ("twin gap") and how much of it simple slate stats
    (own + others' mean & sd) can already explain ("low-order share")."""
    from sklearn.ensemble import HistGradientBoostingRegressor as GBR
    rows = []
    for seed in seeds:
        theta, A = truth(ds, seed)
        rng = np.random.default_rng([99, DATASETS[ds]["id"], seed])
        X = rng.normal(size=(n_slates, N_ITEMS, D)); c = context_raw(X, ds, A)
        cc = c - c.mean(1, keepdims=True)                       # only within-slate differences matter
        S = X.sum(1, keepdims=True); S2 = (X ** 2).sum(1, keepdims=True)
        m = (S - X) / (N_ITEMS - 1); sd = np.sqrt(np.clip((S2 - X ** 2) / (N_ITEMS - 1) - m ** 2, 0, None))
        f_own = X.reshape(-1, D); f_low = np.concatenate([X, m, sd], -1).reshape(-1, 3 * D)
        h = len(f_own) // 2                                     # fit on half, predict on other half
        pred = {}
        for name, f in [("own", f_own), ("low", f_low)]:
            g = GBR(max_iter=300, learning_rate=0.1).fit(f[:h], cc.reshape(-1)[:h])
            pred[name] = g.predict(f[h:]).reshape(-1, N_ITEMS)
        Xh, cch = X.reshape(-1, D)[h:].reshape(-1, N_ITEMS, D), cc.reshape(-1)[h:].reshape(-1, N_ITEMS)
        r2 = {k: 1 - ((cch - v) ** 2).mean() / (cch ** 2).mean() for k, v in pred.items()}
        base = Xh @ theta; sign = DATASETS[ds].get("sign", 1.0)

        def ce(u_model, u_true):
            lp = u_model - u_model.max(1, keepdims=True); lp -= np.log(np.exp(lp).sum(1, keepdims=True))
            pt = np.exp(u_true - u_true.max(1, keepdims=True)); pt /= pt.sum(1, keepdims=True)
            return -(pt * lp).sum(1).mean()
        for lam in lams:
            ut = base + sign * lam * cch
            o = ce(ut, ut)
            rows.append(dict(seed=seed, lam=lam, r2_own=r2["own"], r2_low=r2["low"],
                             gap_twin=ce(base + sign * lam * pred["own"], ut) - o,
                             gap_low=ce(base + sign * lam * pred["low"], ut) - o,
                             gap_plain=ce(base, ut) - o))
    return rows


# ------------------------------------------------------------------ models (input X: (B, n, D) -> logits (B, n))
def mlp(din, w=64, out=1):
    return nn.Sequential(nn.Linear(din, w), nn.ReLU(), nn.Linear(w, w), nn.ReLU(), nn.Linear(w, out))

class PerItem(nn.Module):                       # no context at all
    def __init__(self, w=64): super().__init__(); self.f = mlp(D, w)
    def forward(self, X): return self.f(X).squeeze(-1)

def others_stats(X):
    n = X.shape[1]; S = X.sum(1, keepdims=True); S2 = (X ** 2).sum(1, keepdims=True)
    m = (S - X) / (n - 1); var = ((S2 - X ** 2) / (n - 1) - m ** 2).clamp_min(0)
    return m, (var + 1e-6).sqrt()

class PoolMean(nn.Module):                      # ladder rung 1: own + mean of others
    def __init__(self, w=64): super().__init__(); self.f = mlp(2 * D, w)
    def forward(self, X): m, _ = others_stats(X); return self.f(torch.cat([X, m], -1)).squeeze(-1)

class PoolMeanSD(nn.Module):                    # ladder rung 2: + spread of others
    def __init__(self, w=64): super().__init__(); self.f = mlp(3 * D, w)
    def forward(self, X): m, s = others_stats(X); return self.f(torch.cat([X, m, s], -1)).squeeze(-1)

class SelfAttn(nn.Module):                      # thesis model (post-LN encoder, as in notebook)
    def __init__(self, width=32, heads=2, ffn=64, layers=1, self_only=False):
        super().__init__(); self.self_only = self_only
        self.embed = nn.Linear(D, width); self.ln0 = nn.LayerNorm(width)
        self.blocks = nn.ModuleList([nn.ModuleDict(dict(
            attn=nn.MultiheadAttention(width, heads, batch_first=True), ln1=nn.LayerNorm(width),
            ffn=nn.Sequential(nn.Linear(width, ffn), nn.ReLU(), nn.Linear(ffn, width)), ln2=nn.LayerNorm(width)))
            for _ in range(layers)])
        self.out = nn.Linear(width, 1)
    def forward(self, X):
        h = self.ln0(self.embed(X)); n = X.shape[1]
        mask = ~torch.eye(n, dtype=torch.bool, device=X.device) if self.self_only else None
        for b in self.blocks:
            h = b["ln1"](h + b["attn"](h, h, h, attn_mask=mask, need_weights=False)[0])
            h = b["ln2"](h + b["ffn"](h))
        return self.out(h).squeeze(-1)

# --- DeepHalo: copied unchanged from the authors' DeepHalo-Code/Model/FeatureBased.py
def make_valid_mask(n, lengths, device, mode='without'):
    row = torch.arange(n, device=device).unsqueeze(0)
    if mode == 'with':
        return (row < lengths.unsqueeze(1)) | (row == n - 1)
    else:
        return row < lengths.unsqueeze(1)

def masked_softmax(scores, lengths):
    n = scores.size(1)
    valid = make_valid_mask(n, lengths, scores.device)
    scores = scores.masked_fill(~valid, float('-inf'))
    return F.log_softmax(scores, dim=-1)

class NonlinearTransformation(nn.Module):
    def __init__(self, H, embed=128, dropout=0):
        super().__init__()
        self.fc1 = nn.Linear(embed, embed * H)
        self.fc2 = nn.Linear(embed, embed)
        self.H = H
        self.embed = embed
        self.dropout = nn.Dropout(dropout)
        self.enc_norm = nn.LayerNorm(embed)
    def forward(self, X):
        B, n, _ = X.shape
        X = self.fc1(X).view(B, n, self.H, self.embed)
        X = nn.ReLU()(X)
        X = self.dropout(X)
        X = self.fc2(X)
        X = self.enc_norm(X)
        return X

class DeepHalo(nn.Module):
    def __init__(self, n, input_dim, H, L, embed=128, dropout=0):
        super().__init__()
        self.basic_encoder = nn.Sequential(
            nn.Linear(input_dim, embed), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed, embed), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed, embed))
        self.enc_norm = nn.LayerNorm(embed)
        self.aggregate_linear = nn.ModuleList([nn.Linear(embed, H) for _ in range(L)])
        self.nonlinear = nn.ModuleList([NonlinearTransformation(H, embed) for _ in range(L)])
        self.H = H
        self.embed = embed
        self.final_linear = nn.Linear(embed, 1)
        self.qualinear1 = nn.Linear(embed, embed)
        self.qualinear2 = nn.Linear(embed, embed)
    def forward(self, X, lengths):
        B, n, _ = X.shape
        Z = self.enc_norm(self.basic_encoder(X))
        X = Z.clone()
        for fc, nt in zip(self.aggregate_linear, self.nonlinear):
            Z_bar = (fc(Z).sum(1) / lengths.unsqueeze(1)).unsqueeze(-1).unsqueeze(1)
            phi = nt(X)
            valid = make_valid_mask(n, lengths, X.device)
            phi = phi * valid.unsqueeze(-1).unsqueeze(-1)
            Z = (phi * Z_bar).sum(2) / self.H + Z
        logits = self.final_linear(Z).squeeze(-1)
        probs = masked_softmax(logits, lengths)
        return probs, logits
# --- end of authors' code

class DH(nn.Module):                            # thin wrapper: full slates, return logits
    def __init__(self, L=4, embed=32, H=8):     # paper's LPMC config: embed 32, H 8, L 4
        super().__init__(); self.net = DeepHalo(N_ITEMS, D, H, L, embed)
    def forward(self, X):
        lengths = torch.full((X.shape[0],), X.shape[1], device=X.device, dtype=torch.float32)
        return self.net(X, lengths)[1]

MODELS = {
    "mlp":         lambda: PerItem(),
    "sa1_self":    lambda: SelfAttn(layers=1, self_only=True),   # blindfolded twin of sa1
    "pool_mean":   lambda: PoolMean(),
    "pool_meansd": lambda: PoolMeanSD(),
    "sa1":         lambda: SelfAttn(layers=1),
    "sa2":         lambda: SelfAttn(layers=2),
    "dh1":         lambda: DH(L=1),
    "dh2":         lambda: DH(L=2),
    "dh3":         lambda: DH(L=3),
    "dh4":         lambda: DH(L=4),                               # DeepHalo "pure": paper config
}

def n_params(m, used_only=True):
    skip = ("qualinear",) if used_only else ()
    return sum(p.numel() for n, p in m.named_parameters() if not any(s in n for s in skip))


# ------------------------------------------------------------------ training / evaluation
def to_tensors(data, dev):
    T = {}
    for s, ix in data["idx"].items():
        T[s] = (torch.tensor(data["X"][ix], device=dev), torch.tensor(data["y"][ix], device=dev),
                torch.tensor(data["P"][ix], device=dev))
    return T

@torch.no_grad()
def evaluate(model, T, split, bs=8192):
    X, y, P = T[split]; model.eval(); nll = ce = acc = 0.0
    for i in range(0, len(y), bs):
        lp = F.log_softmax(model(X[i:i + bs]), -1)
        nll += -lp.gather(1, y[i:i + bs, None]).sum().item()
        ce += -(P[i:i + bs] * lp).sum().item()
        acc += (lp.argmax(1) == y[i:i + bs]).sum().item()
    return dict(nll=nll / len(y), ce=ce / len(y), acc=acc / len(y))

def train(model, T, seed, bs=512, phases=((1e-3, 6), (1e-4, 4)), max_epochs=80, log=None):
    """Adam, early stopping on validation NLL; second phase at lower lr from the best state."""
    g = torch.Generator().manual_seed(seed); X, y, _ = T["train"]; n = len(y)
    best_v, best = evaluate(model, T, "val")["nll"], copy.deepcopy(model.state_dict()); eps = []
    for lr, patience in phases:
        model.load_state_dict(best); opt = torch.optim.Adam(model.parameters(), lr=lr); wait = 0
        for ep in range(max_epochs):
            model.train(); perm = torch.randperm(n, generator=g).to(X.device)
            for i in range(0, n, bs):
                b = perm[i:i + bs]
                loss = F.cross_entropy(model(X[b]), y[b])
                opt.zero_grad(); loss.backward(); opt.step()
            v = evaluate(model, T, "val")["nll"]
            if log: log(f"    lr {lr:g} ep {ep + 1:3d} val {v:.5f}")
            if v < best_v - 1e-5: best_v, best, wait = v, copy.deepcopy(model.state_dict()), 0
            else:
                wait += 1
                if wait >= patience: break
        eps.append(ep + 1)
    model.load_state_dict(best)
    return model, eps


# ------------------------------------------------------------------ sweep runner (resumable)
def run_jobs(datasets, models, seeds, out, dev, shard=(0, 1), verbose=False, save_weights=False):
    """Every (dataset, seed) pair in this shard: generate data once, write the oracle
    file, then train each model. Finished runs (json exists) are skipped, so the
    same command can be re-launched after a crash/timeout on any platform."""
    os.makedirs(f"{out}/results", exist_ok=True)
    if save_weights: os.makedirs(f"{out}/weights", exist_ok=True)
    pairs = [(d, s) for d in datasets for s in seeds][shard[0]::shard[1]]
    for ds, seed in pairs:
        todo = [m for m in models if not os.path.exists(f"{out}/results/{ds}__{m}__s{seed}.json")]
        ofile = f"{out}/results/{ds}__oracle__s{seed}.json"
        if not todo and os.path.exists(ofile): continue
        t0 = time.time(); data = make_data(ds, seed)
        if not os.path.exists(ofile):
            json.dump(dict(dataset=ds, model="oracle", seed=seed, lam=LAMBDA[ds], **oracle_metrics(data)),
                      open(ofile, "w"))
        T = to_tensors(data, dev); gen_s = time.time() - t0; del data
        for m in todo:
            torch.manual_seed(1000 * seed + list(MODELS).index(m))
            model = MODELS[m]().to(dev); t1 = time.time()
            model, eps = train(model, T, seed, log=print if verbose else None)
            r = dict(dataset=ds, model=m, seed=seed, lam=LAMBDA[ds], params=n_params(model),
                     epochs=eps, train_s=round(time.time() - t1, 1), gen_s=round(gen_s, 1),
                     **{f"val_{k}": v for k, v in evaluate(model, T, "val").items()},
                     **{f"test_{k}": v for k, v in evaluate(model, T, "test").items()})
            json.dump(r, open(f"{out}/results/{ds}__{m}__s{seed}.json", "w"))
            if save_weights: torch.save(model.state_dict(), f"{out}/weights/{ds}__{m}__s{seed}.pt")
            print(f"{ds} s{seed} {m:12s} test_ce {r['test_ce']:.5f} nll {r['test_nll']:.5f} "
                  f"params {r['params']} epochs {eps} {r['train_s']}s", flush=True)

def time_models(ds="D3_centre", models=None, dev="cpu", seed=0):
    """Exp 1: data-generation time and seconds per training epoch for each model."""
    t0 = time.time(); data = make_data(ds, seed); gen = time.time() - t0
    T = to_tensors(data, dev); del data
    print(f"data generation ({ds}): {gen:.1f}s   device {dev}   threads {torch.get_num_threads()}", flush=True)
    X, y, _ = T["train"]; n = len(y); rows = []
    for m in (models or list(MODELS)):
        model = MODELS[m]().to(dev); opt = torch.optim.Adam(model.parameters(), 1e-3)
        model.train(); perm = torch.randperm(n).to(dev); t1 = time.time()
        for i in range(0, n, 512):
            b = perm[i:i + 512]; loss = F.cross_entropy(model(X[b]), y[b])
            opt.zero_grad(); loss.backward(); opt.step()
        evaluate(model, T, "val")
        if dev == "cuda": torch.cuda.synchronize()
        rows.append((m, n_params(model), time.time() - t1))
        print(f"  {m:12s} params {rows[-1][1]:6d}   {rows[-1][2]:6.1f} s/epoch", flush=True)
    return rows

def collect(out):
    import pandas as pd
    rows = [json.load(open(f"{out}/results/{f}")) for f in os.listdir(f"{out}/results") if f.endswith(".json")]
    return pd.DataFrame(rows)

def parse_seeds(s):
    a, b = s.split("-") if "-" in s else (s, s)
    return list(range(int(a), int(b) + 1))

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--datasets", default=",".join(DATASETS))
    ap.add_argument("--models", default="")
    ap.add_argument("--timing", action="store_true")
    ap.add_argument("--seeds", default="0-29")
    ap.add_argument("--out", default="synth_out")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--threads", type=int, default=0)
    ap.add_argument("--shard", default="0/1")
    ap.add_argument("--verbose", action="store_true")
    a = ap.parse_args()
    if a.threads: torch.set_num_threads(a.threads)
    dev = ("cuda" if torch.cuda.is_available() else "cpu") if a.device == "auto" else a.device
    i, k = map(int, a.shard.split("/"))
    print(f"device {dev} threads {torch.get_num_threads()} shard {i}/{k}", flush=True)
    if a.timing:
        time_models(models=a.models.split(",") if a.models else None, dev=dev)
    else:
        run_jobs(a.datasets.split(","), a.models.split(","), parse_seeds(a.seeds), a.out, dev, (i, k), a.verbose)

Overwriting synth.py


In [10]:
import importlib, synth, torch
importlib.reload(synth)
print(synth.LAMBDA, {d: s.get("sigma") for d, s in synth.DATASETS.items()})   # check the locked values
dev = "cuda" if torch.cuda.is_available() else "cpu"
rows = synth.time_models("D3_centre", dev=dev)        # D3 has the slowest data generation

{'D0_lcl': 1.312, 'D1_sim': 1.831, 'D2_comp': 0.5087, 'D3_centre': 0.2507} {'D0_lcl': None, 'D1_sim': 0.2, 'D2_comp': 0.2, 'D3_centre': 0.2}
data generation (D3_centre): 14.8s   device cuda   threads 2
  mlp          params   4673      1.6 s/epoch
  sa1_self     params   8865      3.3 s/epoch
  pool_mean    params   5057      1.1 s/epoch
  pool_meansd  params   5441      1.0 s/epoch
  sa1          params   8865      1.8 s/epoch
  sa2          params  17409      2.9 s/epoch
  dh1          params  12265      1.9 s/epoch
  dh2          params  22097      2.6 s/epoch
  dh3          params  31929      3.3 s/epoch
  dh4          params  41761      4.4 s/epoch


In [5]:
import synth, importlib, pandas as pd, numpy as np
importlib.reload(synth)

lams = {"D0_lcl":    [0.25, 0.5, 1, 2, 4],
        "D1_sim":    [0.25, 0.5, 1, 2, 4],
        "D2_comp":   [0.1, 0.2, 0.4, 0.8, 1.6],
        "D3_centre": [0.03, 0.06, 0.12, 0.25, 0.5, 1.0]}
cal = []
for ds, L in lams.items():
    print("calibrating", ds, flush=True)
    for r in synth.calibrate(ds, L):
        cal.append(dict(dataset=ds, **r))
cal = pd.DataFrame(cal)
cal.to_csv("calibration.csv", index=False)

calibrating D0_lcl
calibrating D1_sim
calibrating D2_comp
calibrating D3_centre


In [6]:
s = cal.groupby(["dataset", "lam"]).mean(numeric_only=True).drop(columns="seed")
s["low_share_%"] = 100 * (1 - s.gap_low / s.gap_twin)
print(s[["gap_plain", "gap_twin", "gap_low", "low_share_%", "r2_own", "r2_low"]].round(4).to_string())

target = 0.05
print("\nsuggested lambda (twin gap = 0.05 nats):")
for ds, g in s.reset_index().groupby("dataset"):
    lam = np.interp(np.log(target), np.log(g.gap_twin.clip(1e-9)), g.lam)
    print(f"  {ds}: {lam:.3f}")

                gap_plain  gap_twin  gap_low  low_share_%  r2_own  r2_low
dataset   lam                                                            
D0_lcl    0.25     0.0023    0.0021   0.0002      88.5676  0.0764  0.8932
          0.50     0.0090    0.0084   0.0010      88.5710  0.0764  0.8932
          1.00     0.0352    0.0331   0.0038      88.5759  0.0764  0.8932
          2.00     0.1315    0.1248   0.0142      88.6274  0.0764  0.8932
          4.00     0.4171    0.4006   0.0456      88.6182  0.0764  0.8932
D1_sim    0.25     0.0063    0.0048   0.0038      19.3855  0.2141  0.3757
          0.50     0.0239    0.0180   0.0145      19.5017  0.2141  0.3757
          1.00     0.0832    0.0617   0.0498      19.2303  0.2141  0.3757
          2.00     0.2396    0.1758   0.1468      16.4846  0.2141  0.3757
          4.00     0.5222    0.3953   0.3727       5.7273  0.2141  0.3757
D2_comp   0.10     0.0340    0.0150   0.0044      70.5743  0.4990  0.8447
          0.20     0.1387    0.0615   

In [7]:
import importlib, synth, numpy as np, pandas as pd
importlib.reload(synth)

base_lam = {"D1_sim": 0.915, "D2_comp": 0.185, "D3_centre": 0.057}   # from sigma = 0.5
target, out = 0.05, []
for sig in [0.5, 0.3, 0.2, 0.15]:
    for ds, l0 in base_lam.items():
        synth.DATASETS[ds]["sigma"] = sig
        grid = [l0 * (0.5 / sig) ** 2 * f for f in (0.25, 0.5, 1, 2, 4)]   # c shrinks ~ sigma^2
        g = pd.DataFrame(synth.calibrate(ds, grid)).groupby("lam").mean(numeric_only=True).reset_index()
        g["low_share"] = 100 * (1 - g.gap_low / g.gap_twin)
        lg = np.log(g.gap_twin.clip(1e-9))
        out.append(dict(dataset=ds, sigma=sig,
                        lam_at_005=np.interp(np.log(target), lg, g.lam),
                        low_share_at_005=np.interp(np.log(target), lg, g.low_share),
                        r2_own=g.r2_own.iloc[0], r2_low=g.r2_low.iloc[0]))
        print(out[-1], flush=True)
sweep = pd.DataFrame(out); sweep.to_csv("sigma_sweep.csv", index=False)
print(sweep.round(3).to_string())

{'dataset': 'D1_sim', 'sigma': 0.5, 'lam_at_005': np.float64(0.8928959291355809), 'low_share_at_005': np.float64(19.47920114715878), 'r2_own': np.float64(0.21402130151198337), 'r2_low': np.float64(0.37585303937361364)}
{'dataset': 'D2_comp', 'sigma': 0.5, 'lam_at_005': np.float64(0.1816473637316422), 'low_share_at_005': np.float64(70.45008885013502), 'r2_own': np.float64(0.4989716048772502), 'r2_low': np.float64(0.8453961887928738)}
{'dataset': 'D3_centre', 'sigma': 0.5, 'lam_at_005': np.float64(0.05656416619206835), 'low_share_at_005': np.float64(84.00980614914799), 'r2_own': np.float64(0.5152798893776227), 'r2_low': np.float64(0.8937452505323975)}
{'dataset': 'D1_sim', 'sigma': 0.3, 'lam_at_005': np.float64(1.6012007996156692), 'low_share_at_005': np.float64(6.685781029082182), 'r2_own': np.float64(0.09263333839864019), 'r2_low': np.float64(0.1648783310860885)}
{'dataset': 'D2_comp', 'sigma': 0.3, 'lam_at_005': np.float64(0.35229731045415313), 'low_share_at_005': np.float64(48.371894

In [8]:
import importlib, synth, numpy as np, pandas as pd
importlib.reload(synth)

SIG, TARGET_HARD = 0.2, 0.03
start = {"D1_sim": 2.637, "D2_comp": 0.549, "D3_centre": 0.210}    # lam giving twin gap 0.05 at sigma 0.2
lam_final = {"D0_lcl": 1.312}
for ds, l0 in start.items():
    synth.DATASETS[ds]["sigma"] = SIG
    grid = [l0 * f for f in (0.5, 0.75, 1, 1.5, 2, 3, 4)]
    g = pd.DataFrame(synth.calibrate(ds, grid)).groupby("lam").mean(numeric_only=True).reset_index()
    g["low_share"] = 100 * (1 - g.gap_low / g.gap_twin)
    print(ds); print(g[["lam", "gap_twin", "gap_low", "low_share"]].round(4).to_string(index=False))
    lam_final[ds] = float(np.interp(np.log(TARGET_HARD), np.log(g.gap_low), g.lam))
    print(f"  -> lambda for hard part {TARGET_HARD}: {lam_final[ds]:.3f}\n", flush=True)

print("PASTE INTO synth.py:")
print(f'LAMBDA = {{{", ".join(f"{k!r}: {v:.4f}" for k, v in lam_final.items())}}}')
print("and set sigma=0.2 for D1_sim, D2_comp, D3_centre in DATASETS")

D1_sim
    lam  gap_twin  gap_low  low_share
 1.3185    0.0190   0.0184     3.0340
 1.9778    0.0355   0.0345     2.8336
 2.6370    0.0527   0.0514     2.5209
 3.9555    0.0846   0.0833     1.5060
 5.2740    0.1119   0.1120    -0.0589
 7.9110    0.1571   0.1645    -4.7071
10.5480    0.1972   0.2186   -10.8355
  -> lambda for hard part 0.03: 1.831

D2_comp
   lam  gap_twin  gap_low  low_share
0.2745    0.0115   0.0079    31.1342
0.4118    0.0277   0.0192    30.7686
0.5490    0.0519   0.0361    30.4038
0.8235    0.1234   0.0867    29.7358
1.0980    0.2197   0.1556    29.1660
1.6470    0.4449   0.3194    28.2026
2.1960    0.6654   0.4837    27.3038
  -> lambda for hard part 0.03: 0.509

D3_centre
   lam  gap_twin  gap_low  low_share
0.1050    0.0123   0.0048    60.8192
0.1575    0.0296   0.0116    60.6680
0.2100    0.0550   0.0217    60.5997
0.3150    0.1275   0.0502    60.6789
0.4200    0.2204   0.0862    60.8872
0.6300    0.4251   0.1651    61.1707
0.8400    0.6186   0.2409    61.0568
 